# ICS 2405: Knowledge-Based Systmes

## resampling - making `more` from `less`:
> evaluationg models continued

if we content ourselves with a **single train-test** split, that single step provides and
determines both the data we can train from and our testing environment. however, we might get (un)lucky and get a very good train-test split.
eg a very hard training data and very easy testing data. this strategy for evaluation, is aimed at generating multiple estimates for evaluation - that would need multiple datasets. we find a way to make one dataset to be more than enough regardless of how little it could be.

## 1 - cross-validation:

cross-validation takes a number of *folds* eg for a 3-fold cross-validation, we’ll take an entire set of
labeled data and shake it up. we’ll let the data fall randomly—as evenly as possible—into
the three categories (buckets): BI, BII, and BIII.

<div style="display: flex;">
    <img src="cross-bukects.png" alt="cross-validation buckets" style="margin-right: 20px; width: 500px; height: auto;"/>
    <img src="train-test-split.png" alt="train-test split" style="width: 500px; height: auto;"/>
</div>


#### how cross-validation works:
> tldr; we use each cross-validation bucket, in turn, as our test data. we train
on the remainder.

1. Take bucket BI and put it to the side. Put BII and BIII together and use them as our
training set. Train ModelOne — from that combined training set. evaluate ModelOne on bucket BI and record the performance as EvalOne.
2. Take BII and put it to the side. Put buckets BI and BIII together and use them as training set. Train ModelTwo from that combined training set. evaluate ModelTwo on bucket BII and record the performance as EvalTwo.

3. Take bucket BIII and put it to the side. Put BI and BII together as our training set.
Train ModelThree from that combined training set. Now, evaluate ModelThree on BIII
and record the performance as EvalThree ..



### putting it together and `sklearn`:
we’ve recorded three performance values. it takes a a few vlaues we can do graphing them and summarizing them statistically. 
- graphing them can help us understand how variable our performance is with respect to different training and
testing datasets. it tells us something about how our model, our training sets, and our
testing sets interact. with a very wide spread in our performance measures, we would
be justified in being skeptical of any single performance score for our system.
- if the scores are all similar, we have some certainty that, regardless of the specific
train-test split, our system’s performance will be similar.

> ##### caveat + `sklearn` + `k-fold cross-validation`:
> the random sampling
is done without replacement and the train-test splits are all dependent on each other. This
breaks some of the usual assumptions we make.  
> what we’ve described is 3-fold cross-validation. this technique is
k-fold cross-validation — k-fold CV or just k-CV for short. the amount
of cross-validation we do depends on a few factors including the amount of data we have.
3-, 5-, and 10-fold CV are commonly used and recommended.
> 

In [1]:
# setup
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
from sklearn import datasets, model_selection as skms, linear_model, metrics, neighbors

diabetes = datasets.load_diabetes()

##### a 5-fold CV with sklearn:  

In [3]:
# data, model, fit & cv-score
model = neighbors.KNeighborsRegressor(10)
skms.cross_val_score(model, diabetes.data, diabetes.target, cv=5, scoring='neg_mean_squared_error')

# notes:
# defaults for cross_val_score are
# cv=3 fold, no shuffle, stratified if classifier
# model.score by default (regressors: r2, classifiers: accuracy)

array([-3206.7541573 , -3426.43134831, -3587.94215909, -3039.49443182,
       -3282.60159091])

### *note*:
> the default value for the `cv` argument to `cross_val_score` is `None`.
> cv: int or None or others. determines the cross-validation splitting strategy.
possible inputs for cv are:
> - None, to use the default 3-fold cross validation,
> - Integer, to specify the number of folds in a (Stratified)KFold,
> - Others.  
> for or integer/None inputs, if the estimator is a **classifier** and y is either binary or
multiclass, `StratifiedKFold` is used. in all other cases, `KFold` is used

### take away:
(1) by default we’re doing 3-fold CV and  
(2) for classification problems, sklearn uses *stratification*.

## 2 - stratification: 
we can have cross-validation in a classification context. here, we tell
`cross_val_score` to use 5-fold CV:

In [5]:
iris = datasets.load_iris()
model = neighbors.KNeighborsClassifier(10)
skms.cross_val_score(model, iris.data, iris.target, cv=5) # scoring = stratifiedkfold by default coz this is a classifier

array([0.96666667, 1.        , 1.        , 0.93333333, 1.        ])

**stratification** means that when we make our *training-testing* splits for cross-validation, we want to respect the
proportions of the targets that are present in our data.  

#### example two-fold:

In [6]:
# not stratified
pet = np.array(['cat', 'dog', 'cat', 'dog', 'dog', 'dog'])

list_folds = list(skms.KFold(2).split(pet))

training_idxs = np.array(list_folds)[:, 0, :]
print(pet[training_idxs])

[['dog' 'dog' 'dog']
 ['cat' 'dog' 'cat']]


#### problem: 
notice that there were no cats in the first fold. that’s not
great. if that were our target, we would have no examples to learn about cats. that is a problem - that can’t be good. **stratified sampling** enforces fair play among the cats and dogs:

In [7]:
# stratified
# note: typically this is behind the scenes
# making StratifiedKFold produce readable output
# requires some trickery. feel free to ignore.
pet = np.array(['cat', 'dog', 'cat', 'dog', 'dog', 'dog'])

idxs = np.array(list(skms.StratifiedKFold(2).split(np.ones_like(pet), pet)))

training_idxs = idxs[:, 0, :]

print(pet[training_idxs])

[['cat' 'dog' 'dog']
 ['cat' 'dog' 'dog']]


#### verdict - stratification
both folds have a balanced number of cats and dogs, equal to their proportion in
the overall dataset.  

stratification ensures that we have the same (or nearly the same, once
we round off uneven splits) percent of dogs and cats in each of our training sets as we do in
our entire, available population. without stratification, we could end up having too few
(or even none) of a target class - we don’t expect that training data to lead to a good model.
stratification is particularly useful when:  
1. we have limited data overall or
2. we have classes that are poorly represented in our dataset.

Poor representation might be due to
rareness—it
might be due to our data collection processes. having a limited total amount of data makes
everything rare, in a sense.
how does the default stratification apply to the iris dataset? it means that
when we perform the cross-validation splits, we can be sure that each of the training sets
has a balanced representation from each of the three possible target flowers. what if we
don’t want stratification? it’s slightly more tricky, but we can do it:

In [8]:
# running nonstratified CV
iris = datasets.load_iris()
model = neighbors.KNeighborsClassifier(10) 

non_strat_kf = skms.KFold(5)

skms.cross_val_score(model, iris.data, iris.target, cv=non_strat_kf)

array([1.        , 1.        , 0.86666667, 0.96666667, 0.76666667])

### 3 - repeated train-test splits:

> will continue from here

### end 
> see the resampling notebook for more